# SciAgent on Google Colab
Run the cells top to bottom. **Runtime → Change runtime type → T4 GPU** speeds up embeddings and TensorFlow (optional; CPU works).

API key: click the 🔑 *Secrets* icon in the left sidebar, add `ANTHROPIC_API_KEY` (or `OPENAI_API_KEY`) and enable notebook access. Without a key the offline baseline agent is used.

## 1. Get the code
Option A: upload `sci-agent.zip`. Option B: mount Google Drive (files persist between sessions — recommended).

In [ ]:
# Option A: upload the zip (files are lost when the runtime resets)
from google.colab import files
up = files.upload()            # choose sci-agent.zip
!unzip -q -o sci-agent.zip -d /content/

In [ ]:
# Option B (recommended): keep the project + data on Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/sciagent && unzip -q -o /content/drive/MyDrive/sci-agent.zip -d /content/drive/MyDrive/sciagent/
# %cd /content/drive/MyDrive/sciagent/sci-agent

In [ ]:
%cd /content/sci-agent
# TensorFlow, scikit-learn, sentence-transformers, matplotlib are preinstalled on Colab
!pip install -q pymupdf pdfplumber pypdf anthropic openai pytest "pillow<13"

In [ ]:
import os
from google.colab import userdata
for k in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY"):
    try:
        os.environ[k] = userdata.get(k)
        print(k, "set")
    except Exception:
        pass
# os.environ["SCIAGENT_LLM"] = "anthropic"   # force a provider: anthropic | openai | offline

## 1b. Optional: local LLM with Ollama (no API key needed)
Runs an open-source model on Colab's free GPU (Runtime → Change runtime type → **T4 GPU**).
Skip this section if you use an Anthropic/OpenAI key. Run it again after every runtime restart.

In [ ]:
# Install Ollama and start the server in the background (~1-2 min)
!sudo apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import os, subprocess, time
env = {**os.environ, "OLLAMA_CONTEXT_LENGTH": "16384"}   # agent conversations need a long context
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env,
                               stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!curl -s http://localhost:11434/api/version

In [ ]:
# Download a model with tool-calling support (~4.7 GB, a few minutes)
MODEL = "qwen2.5:7b"          # alternatives: "llama3.1:8b", "qwen2.5:14b" (slower, better)
!ollama pull {MODEL}

# Point SciAgent's OpenAI-compatible adapter at the local server
os.environ["OPENAI_BASE_URL"] = "http://localhost:11434/v1"
os.environ["OPENAI_API_KEY"]  = "ollama"      # any non-empty string; Ollama ignores it
os.environ["OPENAI_MODEL"]    = MODEL
os.environ["SCIAGENT_LLM"]    = "openai"
os.environ.pop("ANTHROPIC_API_KEY", None)
print("SciAgent will use the local model:", MODEL)

In [ ]:
# Quick check: should print a short answer
from openai import OpenAI
r = OpenAI().chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Say hello in five words."}])
print(r.choices[0].message.content)

## 2. Sanity checks (no network, no key needed)

In [ ]:
!python -m pytest -q tests

In [ ]:
!python eval/run_eval.py --fixture --provider offline --lsa

## 3. ML component — Burgers experiment incl. TensorFlow/Keras surrogate

In [ ]:
!python -m sciagent.ml.burgers --tf
from IPython.display import Image, display
display(Image("results/burgers_surrogate.png"))

## 4. Build the real corpus (arXiv → PDFs → chunks → index)
~10 minutes. If arXiv rate-limits (HTTP 503), just re-run: downloads are resumable.

In [ ]:
!python -m sciagent.collect --max-per-query 25 --keep 40

In [ ]:
!python -m sciagent.ingest

In [ ]:
!python -m sciagent.index

In [ ]:
# Check which pipeline steps have produced their files (run any time)
import os, json
print('working dir:', os.getcwd())
for f in ['data/papers.jsonl', 'data/chunks.jsonl', 'data/index.pkl']:
    if os.path.exists(f):
        n = sum(1 for _ in open(f)) if f.endswith('.jsonl') else os.path.getsize(f)
        print(f'OK      {f}  ({n} lines)' if f.endswith('.jsonl') else f'OK      {f}  ({n/1e6:.1f} MB)')
    else:
        print(f'MISSING {f}')
print('PDFs downloaded:', len(os.listdir('data/pdfs')) if os.path.isdir('data/pdfs') else 0)

## 5. Ask the agent

In [ ]:
!python -m sciagent.agent "Do machine-learning solvers for fluid PDEs really deliver the large speedups that are often reported?" --trace

In [ ]:
!python -m sciagent.agent "In the Fourier Neural Operator paper, what does the figure comparing models on the Navier-Stokes equation show?" --trace

In [ ]:
!python -m sciagent.agent "What drag coefficient do the papers report for a hypersonic re-entry capsule at Mach 25?" --trace

## 6. Evaluation + examples

In [ ]:
!python eval/run_eval.py --judge          # LLM agent (needs key); add --provider offline for the baseline
!python eval/run_eval.py --provider offline --no-robustness
!python examples/run_examples.py

In [ ]:
from IPython.display import Markdown
import glob
for f in sorted(glob.glob("results/eval_arxiv_*.md")):
    display(Markdown(open(f).read()))

## 7. Download results

In [ ]:
!zip -qr results_bundle.zip results examples
from google.colab import files
files.download("results_bundle.zip")